# Mixer-SNN vs DMP-SNN — benchmark trên Colab GPU

Chạy lần lượt từ trên xuống. Mỗi lần chạy xong một ô, **lưu notebook (Ctrl+S)** để Claude đọc được output.

* Kết quả (`results/*.json`) và checkpoint lưu trên Google Drive → Colab ngắt kết nối thì chạy lại đúng ô đó, run dang dở sẽ **tự resume**, run đã xong sẽ **tự bỏ qua**.
* Thứ tự ưu tiên: A (bảng chính, seed 0) → B (delayed recall) → C (ablation) → D (thêm seed 1, 2).
* Ô cuối cùng in toàn bộ kết quả dạng gọn để dán/đọc lại.

In [10]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

: 

In [11]:
# Lưu kết quả lên Google Drive (nếu mount lỗi trong VS Code thì kết quả nằm ở /content/bench_out — vẫn đọc được qua ô cuối)
import os
OUT = '/content/bench_out'
DATA = '/content/data'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/XLDL_bench'
except Exception as e:
    print('Drive mount failed:', e)
os.makedirs(OUT, exist_ok=True)
DATA = '/content/data'
print('OUT =', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT = /content/drive/MyDrive/XLDL_bench


In [12]:
import os
if 'OUT' not in globals():
    OUT, DATA = '/content/bench_out', '/content/data'
    os.makedirs(OUT, exist_ok=True)
REPO = 'https://github.com/incognito-tab/XLDL.git'
REPO_DIR = '/content/XLDL'
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1
!pip -q install spikingjelly==0.0.0.0.14 einops ml_collections timm h5py

import shlex, subprocess, time

def run(args, time_limit=0, out=None):
    """Chạy một thí nghiệm; in log theo từng epoch."""
    cmd = f'python -m bench.run {args} --out {out or OUT} --data-root {DATA}'
    if time_limit:
        cmd += f' --time-limit {time_limit}'
    print('\n$', cmd, flush=True)
    t = time.time()
    p = subprocess.Popen(shlex.split(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'warn' not in line.lower():
            print(line, end='', flush=True)
    code = p.wait()
    print(f'[exit {code}] {(time.time() - t) / 60:.1f} min', flush=True)
    return code

MAIN = [  # (dataset, model args)
    ('fmnist',        '--model mixer --mixer-size small --T 4'),
    ('fmnist',        '--model mixer --mixer-size lite --T 4'),
    ('fmnist',        '--model dmp'),
    ('fmnist_stream', '--model mixer --mixer-size lite'),
    ('fmnist_stream', '--model dmp'),
    ('cifar10',       '--model mixer --mixer-size small --T 4'),
    ('cifar10',       '--model mixer --mixer-size lite --T 4'),
    ('cifar10',       '--model dmp'),
    ('shd',           '--model mixer --mixer-size lite'),
    ('shd',           '--model dmp'),
]
DELAYS = [10, 25, 50, 100, 200]
print('ready: run(), MAIN, DELAYS')


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.49 KiB | 1.49 MiB/s, done.
From https://github.com/incognito-tab/XLDL
   0bcfb53..05ed730  main       -> origin/main
Updating 0bcfb53..05ed730
Fast-forward
 bench/colab_run.ipynb | 97 +++++++++++++++++++++++++++++++++++++++------------
 1 file changed, 75 insertions(+), 22 deletions(-)
05ed730 (HEAD -> main, origin/main, origin/HEAD) Restore run()/MAIN definitions in Colab notebook setup cell
ready: run(), MAIN, DELAYS


## 0. Smoke test (~5–10 phút)
Mỗi cấu hình chạy 1 epoch trên 512 mẫu để chắc mọi thứ chạy được trên GPU, đồng thời tải dữ liệu (CIFAR-10 ~170MB, SHD ~170MB). **Xem cột `s/epoch`** để ước lượng thời gian thật (nhân với số mẫu thật / 512).

In [13]:
SMOKE = '/content/smoke'
for ds, m in MAIN:
    run(f'--dataset {ds} {m} --epochs 1 --max-train 512', out=SMOKE)
run('--dataset recall --delay 50 --model mixer --mixer-size lite --epochs 1 --max-train 512', out=SMOKE)
run('--dataset recall --delay 50 --model dmp --epochs 1 --max-train 512', out=SMOKE)


$ python -m bench.run --dataset fmnist --model mixer --mixer-size small --T 4 --epochs 1 --max-train 512 --out /content/smoke --data-root /content/data

  0%|          | 0.00/26.4M [00:00<?, ?B/s]
  0%|          | 32.8k/26.4M [00:00<01:55, 229kB/s]
  0%|          | 65.5k/26.4M [00:00<01:56, 227kB/s]
  0%|          | 131k/26.4M [00:00<01:19, 330kB/s] 
  1%|          | 229k/26.4M [00:00<00:56, 467kB/s]
  2%|▏         | 459k/26.4M [00:00<00:29, 868kB/s]
  3%|▎         | 786k/26.4M [00:00<00:19, 1.34MB/s]
  6%|▌         | 1.57M/26.4M [00:01<00:09, 2.67MB/s]
 12%|█▏        | 3.18M/26.4M [00:01<00:04, 5.34MB/s]
 24%|██▍       | 6.36M/26.4M [00:01<00:01, 10.5MB/s]
 40%|███▉      | 10.5M/26.4M [00:01<00:01, 15.8MB/s]
 55%|█████▍    | 14.4M/26.4M [00:01<00:00, 19.3MB/s]
 70%|███████   | 18.5M/26.4M [00:01<00:00, 21.9MB/s]
 86%|████████▌ | 22.7M/26.4M [00:01<00:00, 23.7MB/s]
100%|█████████▉| 26.4M/26.4M [00:02<00:00, 24.2MB/s]
100%|██████████| 26.4M/26.4M [00:02<00:00, 12.9MB/s]

  0%|         

0

## A. Bảng chính — seed 0

In [ ]:
for ds, m in MAIN:
    run(f'--dataset {ds} {m} --seed 0')


$ python -m bench.run --dataset fmnist --model mixer --mixer-size small --T 4 --seed 0 --out /content/drive/MyDrive/XLDL_bench --data-root /content/data
=== fmnist__mixer-small_T4__s0 | params 24,514,694 | train 54000 val 6000 test 10000 | cuda
ep   1/15 | train 0.7433 loss 0.7136 | val 0.8367 | best 0.8367@1 | 225s
ep   2/15 | train 0.8764 loss 0.3356 | val 0.8732 | best 0.8732@2 | 219s
ep   3/15 | train 0.8984 loss 0.2750 | val 0.8887 | best 0.8887@3 | 225s
ep   4/15 | train 0.9102 loss 0.2426 | val 0.8963 | best 0.8963@4 | 226s
ep   5/15 | train 0.9214 loss 0.2148 | val 0.8945 | best 0.8963@4 | 223s
ep   6/15 | train 0.9320 loss 0.1861 | val 0.8995 | best 0.8995@6 | 222s
ep   7/15 | train 0.9378 loss 0.1667 | val 0.8997 | best 0.8997@7 | 223s
ep   8/15 | train 0.9461 loss 0.1414 | val 0.9053 | best 0.9053@8 | 225s
ep   9/15 | train 0.9553 loss 0.1196 | val 0.9105 | best 0.9105@9 | 227s
ep  10/15 | train 0.9641 loss 0.0964 | val 0.9067 | best 0.9105@9 | 224s
ep  11/15 | train 0.9709

## B. Delayed recall — nhớ quá khứ theo độ trễ D
* `dmp` mặc định: cửa sổ nhớ `theta` = độ dài chuỗi (luôn đủ dài).
* `dmp --theta 50 --tag fixedtheta`: cửa sổ nhớ cố định 50 bước → kiểm tra điều gì xảy ra khi D > theta.

In [ ]:
for seed in (0, 1):
    for D in DELAYS:
        run(f'--dataset recall --delay {D} --model dmp --seed {seed}')
        run(f'--dataset recall --delay {D} --model dmp --theta 50 --tag fixedtheta --seed {seed}')
        run(f'--dataset recall --delay {D} --model mixer --mixer-size lite --seed {seed}')

## C. Ablation (seed 0)
* DMP **không có bộ nhớ chậm** (`--gamma 0`, train lại từ đầu) → bộ nhớ đóng góp bao nhiêu.
* DMP: độ nhạy với `d` (số chiều bộ nhớ) và `theta` (cửa sổ nhớ) trên SHD.
* Mixer: tăng hằng số thời gian LIF `tau` 2 → 8 (nhớ lâu hơn có giúp Mixer không?) và số bước `T` trên ảnh tĩnh.

In [ ]:
ABL = [
    '--dataset shd --model dmp --gamma 0',
    '--dataset fmnist_stream --model dmp --gamma 0',
    '--dataset recall --delay 50 --model dmp --gamma 0',
    *[f'--dataset shd --model dmp --d {d}' for d in (1, 2, 5, 20)],
    *[f'--dataset shd --model dmp --theta {t}' for t in (10, 20, 80, 100)],
    '--dataset shd --model mixer --mixer-size lite --tau 8',
    '--dataset fmnist_stream --model mixer --mixer-size lite --tau 8',
    '--dataset recall --delay 50 --model mixer --mixer-size lite --tau 8',
    *[f'--dataset cifar10 --model mixer --mixer-size lite --T {T}' for T in (1, 2, 8)],
]
for a in ABL:
    run(a + ' --seed 0')

## D. Thêm seed 1, 2 cho bảng chính (để có mean ± std)

In [ ]:
for seed in (1, 2):
    for ds, m in MAIN:
        run(f'--dataset {ds} {m} --seed {seed}')

## Tổng hợp

In [ ]:
!cd {REPO_DIR} && python -m bench.summarize --out {OUT}
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUT}/plots/*.png')):
    print(p); display(Image(p))

In [ ]:
# In toàn bộ kết quả dạng gọn (để Claude đọc trực tiếp từ notebook)
import json, glob
for p in sorted(glob.glob(f'{OUT}/results/*.json')):
    r = json.load(open(p))
    h = r.pop('history'); r.pop('args', None)
    r['val_curve'] = [round(e['val_acc'] * 100, 1) for e in h]
    r['train_curve'] = [round(e['train_acc'] * 100, 1) for e in h]
    print('RESULTJSON', json.dumps(r))